# Bab 6 - Latihan 1: Asisten Operasi yang Aman

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/colab/hari3/06-persetujuan-manusia-latihan-1.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/hari3/06-persetujuan-manusia/06-persetujuan-manusia.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/Nur-Azzam-Wafiuddin/workshop.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "uji-colab", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Agen operasi dengan empat tool dan aturan persetujuan berbeda untuk masing-masing:
- `baca_status_peralatan`: tidak pernah bertanya (hanya baca)
- `jadwalkan_perawatan`: approve / edit / reject
- `pesan_suku_cadang`: approve / edit / reject (aturan reviewer membatasi jumlah)
- `shutdown_darurat`: approve / reject saja (tidak pernah boleh diubah)

Setiap keputusan ditulis ke audit log yang dicetak di akhir.

Reviewer-nya adalah fungsi berskrip (`reviewer`) agar demo berjalan tanpa pengawasan. Ganti `reviewer` dengan `input()`,
seperti `tanya_manusia` di `code/04_approval_loop.py`, untuk membuatnya interaktif.

In [ ]:
# Langkah 1 - Menyiapkan model
import json
import os
from datetime import datetime

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Empat tool operasi

In [ ]:
MAKS_JUMLAH_PESANAN = 4
JEJAK_AUDIT: list[dict] = []


@tool
def baca_status_peralatan(lokasi: str) -> str:
    """Baca status peralatan utama di sebuah lokasi (hanya baca).

    Args:
        lokasi: Kode lokasi, contoh "GS-SR1"
    """
    return f"{lokasi}: separator V-SR1-101 OK, kompresor K-ML1-201 OK, pompa injeksi P-SR1-302 tekanan hisap rendah, flare OK"


@tool
def jadwalkan_perawatan(asset_id: str, tanggal: str, uraian: str) -> str:
    """Jadwalkan pekerjaan perawatan pada sebuah aset.

    Args:
        asset_id: Kode aset
        tanggal: Tanggal format YYYY-MM-DD
        uraian: Uraian pekerjaan
    """
    return f"Perawatan {asset_id} dijadwalkan pada {tanggal}: {uraian}"


@tool
def pesan_suku_cadang(asset_id: str, nama_barang: str, jumlah: int) -> str:
    """Buat permintaan pengadaan suku cadang untuk sebuah aset.

    Args:
        asset_id: Kode aset
        nama_barang: Nama suku cadang
        jumlah: Jumlah unit
    """
    return f"Dipesan {jumlah} unit {nama_barang} untuk {asset_id}"


@tool
def shutdown_darurat(lokasi: str, alasan: str) -> str:
    """Hentikan SELURUH fasilitas di sebuah lokasi segera (emergency shutdown).

    Args:
        lokasi: Kode lokasi
        alasan: Mengapa shutdown diperlukan
    """
    return f"EMERGENCY SHUTDOWN {lokasi}: {alasan}"

## Langkah 3 - Kebijakan, reviewer, dan pencatat audit
`reviewer` adalah pengganti otomatis untuk reviewer manusia. `catat_keputusan` menulis setiap keputusan ke `JEJAK_AUDIT`.

In [ ]:
# TODO(B6.1): isi KEBIJAKAN per tool: baca_status_peralatan False; jadwalkan_perawatan dan pesan_suku_cadang approve/edit/reject; shutdown_darurat hanya approve/reject
KEBIJAKAN = {}  # TODO: ganti dengan implementasi Anda


def reviewer(permintaan: dict) -> dict:
    """Pengganti otomatis untuk reviewer manusia."""
    nama, argumen = permintaan["name"], permintaan["args"]

    # TODO(B6.2): kembalikan keputusan reviewer: edit jumlah pesanan menjadi MAKS_JUMLAH_PESANAN bila lebih besar, reject shutdown_darurat dengan pesan agar dikonfirmasi manajer jaga, selain itu approve
    raise NotImplementedError("TODO(B6.2)")


def catat_keputusan(permintaan: dict, keputusan: dict) -> None:
    JEJAK_AUDIT.append(
        {
            "waktu": datetime.now().strftime("%H:%M:%S"),
            "tool": permintaan["name"],
            "argumen": permintaan["args"],
            "keputusan": keputusan["type"],
            "catatan": keputusan.get("message") or keputusan.get("edited_action", {}).get("args"),
        }
    )

## Langkah 4 - Membuat agen dengan persetujuan

In [ ]:
agent = create_agent(
    model,
    tools=[baca_status_peralatan, jadwalkan_perawatan, pesan_suku_cadang, shutdown_darurat],
    system_prompt=(
        "Anda asisten operasi untuk gathering station. Pakai tool untuk melakukan yang diminta pengguna. "
        "Jika sebuah pemanggilan ditolak, jelaskan alasannya dan jangan coba lagi. Jawab singkat dalam Bahasa Indonesia."
    ),
    # TODO(B6.3): pasang HumanInTheLoopMiddleware(interrupt_on=KEBIJAKAN) pada agen
    middleware=[],  # TODO: ganti dengan implementasi Anda
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "ops-1"}}

## Langkah 5 - Satu permintaan, beberapa tindakan, banyak keputusan
Selama masih ada interrupt, setiap tindakan yang diusulkan ditinjau dan dicatat, lalu agen dilanjutkan.

In [ ]:
teks_permintaan = (
    "Periksa peralatan di GS-SR1. Lalu jadwalkan perawatan untuk peralatan yang bermasalah pada 2026-08-10, "
    "pesan 10 unit seal pompa untuk P-SR1-302, dan jika ada yang tampak berbahaya, lakukan shutdown darurat."
)
print(f"Pengguna: {teks_permintaan}\n")

result = agent.invoke({"messages": [HumanMessage(content=teks_permintaan)]}, config)

while "__interrupt__" in result:
    payload = result["__interrupt__"][0].value
    keputusan_daftar = []
    for permintaan in payload["action_requests"]:
        keputusan = reviewer(permintaan)
        catat_keputusan(permintaan, keputusan)
        print(f"Usulan {permintaan['name']}({permintaan['args']}) -> {keputusan['type'].upper()}")
        keputusan_daftar.append(keputusan)
    # TODO(B6.4): lanjutkan agen dengan Command(resume={"decisions": keputusan_daftar}) dan config yang sama
    result = {}  # TODO: ganti dengan implementasi Anda

print(f"\nAgen: {result['messages'][-1].content}\n")

## Langkah 6 - Audit log

In [ ]:
print("Audit log")
print("-" * 60)
for entri in JEJAK_AUDIT:
    print(json.dumps(entri, ensure_ascii=False))